# 02 – PHA, FMEA és HAZOP: strukturált veszélyazonosítás

**2. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- PHA hazard list készítése
- FMEA failure mode–effect–cause lánc felépítése
- RPN kiszámítása és korlátainak felismerése
- HAZOP guideword alapú eltérések létrehozása

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Három módszer – három nézőpont

- **PHA:** magas szinten milyen veszélyek vannak?
- **FMEA:** hogyan hibásodhat meg egy elem/funkció, és mi a hatás?
- **HAZOP:** milyen tervezési szándéktól való eltérések jöhetnek létre, mi okozza őket és mi a következmény?

A cél nem három független lista, hanem később összekapcsolható traceability.


## 2. PHA – TK-101 első hazard list

A PHA korai, magas szintű elemzés. Példák: overflow, ignition/fire, loss of containment. A prioritási kategóriák szervezetspecifikusak; itt csak oktatási címkéket használunk.


In [ ]:
pha=pd.DataFrame([
 {"hazard":"Overflow","cause":"Level protection ineffective","consequence":"Release","safeguard":"LSHH+XV+operator","priority":"high"},
 {"hazard":"Fire","cause":"Release + ignition","consequence":"Fire escalation","safeguard":"containment+fire protection","priority":"high"},
 {"hazard":"Loss of containment","cause":"Tank/connection integrity loss","consequence":"Release","safeguard":"inspection+containment","priority":"medium"}])
pha


## 3. FMEA és RPN

Az FMEA tipikus lánca: **function → failure mode → effect → cause → control → rating → action**. Klasszikus oktatási RPN:

$$RPN=S\times O\times D.$$

A skálákat előre definiálni kell. **Fix RPN elfogadási küszöböt nem tekintünk univerzálisnak.** Azonos RPN mögött eltérő severity profil lehet.


In [ ]:
def rpn(S,O,D):
    if not all(1<=int(x)<=10 for x in [S,O,D]): raise ValueError("S/O/D: 1..10")
    return int(S)*int(O)*int(D)

examples=pd.DataFrame([
 {"mode":"A","S":10,"O":2,"D":4},
 {"mode":"B","S":5,"O":4,"D":4},
 {"mode":"C","S":8,"O":5,"D":2}])
examples["RPN"]=examples.apply(lambda r:rpn(r.S,r.O,r.D),axis=1)
examples


### Azonos RPN ≠ azonos kockázati profil

`10×2×4` és `5×4×4` egyaránt 80. Safety kontextusban a severity sokszor önálló prioritást érdemel. Mutassuk meg a severity-first rendezést.


In [ ]:
examples.sort_values(["S","O","D","RPN"],ascending=False)


## 4. TK-101 FMEA

Az S/O/D értékek itt **szintetikus oktatási értékek**, nem üzemi bizonyítékok.


In [ ]:
fmea=pd.DataFrame([
 {"component":"LT-101","failure_mode":"failed low / bias","effect":"high level detected late","S":9,"O":4,"D":6},
 {"component":"LSHH-101","failure_mode":"fail on demand","effect":"trip not initiated","S":10,"O":2,"D":7},
 {"component":"XV-101","failure_mode":"fail on demand","effect":"inflow continues","S":10,"O":2,"D":6},
 {"component":"HMI/operator","failure_mode":"alarm missed","effect":"no timely manual recovery","S":9,"O":3,"D":5}])
fmea["RPN"]=fmea.apply(lambda r:rpn(r.S,r.O,r.D),axis=1)
fmea.sort_values("RPN",ascending=False)


## 5. HAZOP: guideword + parameter → deviation

A `TK-101 filling` node-on a `LEVEL` és `FLOW` paramétereket vizsgáljuk. Guideword példák: NO, MORE, LESS, REVERSE, OTHER THAN. Nem minden kombináció értelmes; a módszer célja a strukturált mérnöki gondolkodás.


In [ ]:
hazop=pd.DataFrame([
 {"node":"TK-101 filling","parameter":"LEVEL","guideword":"MORE","deviation":"High level","causes":"excess inflow; LT failed low; shutdown failure","consequences":"overflow/release","safeguards":"LAH; LSHH; XV; operator"},
 {"node":"TK-101 filling","parameter":"FLOW","guideword":"MORE","deviation":"Excess feed flow","causes":"pump/control fault","consequences":"level rises faster","safeguards":"flow/level monitoring; trip"},
 {"node":"TK-101 shutdown","parameter":"SHUTDOWN","guideword":"NO","deviation":"No effective shutdown","causes":"LSHH/XV fail on demand","consequences":"inflow continues at high level","safeguards":"operator intervention"}])
hazop


## 6. Traceability a három módszer között

A következő lánc már közvetlen bemenetet adhat a Fault Tree Analysishez:

**PHA: Overflow → FMEA: LT failed low / LSHH FOD / XV FOD → HAZOP: MORE LEVEL / NO SHUTDOWN → FTA basic events.**


In [ ]:
trace=pd.DataFrame([
 {"PHA hazard":"Overflow","FMEA mode":"LT-101 failed low","HAZOP deviation":"MORE LEVEL","next":"FTA basic event"},
 {"PHA hazard":"Overflow","FMEA mode":"LSHH fail on demand","HAZOP deviation":"NO SHUTDOWN","next":"FTA basic event"},
 {"PHA hazard":"Overflow","FMEA mode":"XV fail on demand","HAZOP deviation":"NO SHUTDOWN","next":"FTA basic event"}])
trace


## Próbáld ki!

1. Adj FMEA-sort a maintenance overdue állapothoz: komponenshiba vagy szervezeti/process failure mode?
2. Keress két azonos RPN-jű, de eltérő severity-jű sort.
3. Adj új HAZOP sort LESS FLOW vagy REVERSE FLOW guideworddal, ha mérnökileg értelmes.
4. Jelöld, mely safeguardok jelenjenek meg a következő FTA-ban.


## Összefoglalás

PHA veszélyeket, FMEA failure mode-okat, HAZOP eltéréseket strukturál. A következő lépés a kombinációs logika: **mely eseménykombinációk elegendők az overflow top eventhez?**
